In [24]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import numpy as np
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.fit_params.weibull_fit_params import WeibullFitParams

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [25]:
power_law_db = pl.read_parquet(r"MLE_fit_data\PowerLaw").filter(pl.col("q_err").is_not_nan())
weibull_db = pl.read_parquet(r"MLE_fit_data\Weibull").filter(pl.col("k_err").is_not_nan())
log_normal_db = pl.read_parquet(r"MLE_fit_data\LogNormal").filter(pl.col("mu_err").is_not_nan())

In [30]:
aic_db = pl.concat([
    power_law_db.select("aic").with_columns(pl.lit("PL").alias("fit_type")),
    weibull_db.select("aic").with_columns(pl.lit("W").alias("fit_type")),
    log_normal_db.select("aic").with_columns(pl.lit("LN").alias("fit_type"))
]).with_columns(
    (pl.col("aic") - pl.col("aic").min()).alias("delta_aic")
)

aic_db.group_by("fit_type").agg(
    # pl.col("aic").mean().alias("mean_aic"),
    pl.col("aic").std().alias("std_aic"),
    # pl.col("aic").min().alias("min_aic"),
    # pl.col("aic").max().alias("max_aic"),

    (pl.col("aic").mean() - aic_db["aic"].min()).alias("rel_mean_aic"),
    (pl.col("aic").min() - aic_db["aic"].min()).alias("rel_min_aic"),
    (pl.col("aic").max() - aic_db["aic"].min()).alias("rel_max_aic"),
    pl.count()
)


C:\Users\Joshu\AppData\Local\Temp\ipykernel_30820\4261658435.py:18: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count()


fit_type,std_aic,rel_mean_aic,rel_min_aic,rel_max_aic,count
str,f64,f64,f64,f64,u32
"""LN""",3247.698532,5551.774344,0.0,17086.757129,90
"""W""",2055.09042,7267.94055,3392.335998,11025.435595,41
"""PL""",7291.117699,26445.923072,13307.179547,51642.244476,60


In [ ]:
import itertools

types = df["fit_type"].unique().to_list()

for t1, t2 in itertools.combinations(types, 2):
    a = df.filter(pl.col("fit_type") == t1)["aic"].to_numpy()
    b = df.filter(pl.col("fit_type") == t2)["aic"].to_numpy()

    p = (a[:, None] > b).mean()

    print(f"P({t1} > {t2}) = {p:.3f}")
    print(f"P({t2} > {t1}) = {1-p:.3f}")